In [ ]:
import pandas as pd
import yaml
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.rag import rag_for_metric
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap
import matplotlib
matplotlib.use("Agg")

# Load data from CSV
DATA_DIR = Path('..') / 'data'
file_path = DATA_DIR / 'sample.csv'
df = pd.read_csv(file_path)

# Define columns for metrics
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd',
    'period_col': 'QTR',
    'band_col': 'BAND',
}

# Initialize PDMetricsService with desired metrics
metric_ids = ['auc_roc', 'gini', 'ks']
service = PDMetricsService(metric_ids)

# Filter data for development and validation periods
df_dev = df[df['score_year'] == 2019]  # 2019 as development year
df_val = df[df['score_year'] == 2020]  # 2020 as validation year

# Compute metrics separately for both periods
result_dev = service.compute(df_dev, params)
result_val = service.compute(df_val, params)

# Check and print results for Gini, AUROC, and KS
print("Development Metrics:", result_dev)
print("Validation Metrics:", result_val)

# RAG calculation using the validation_rag.yaml configuration
rag_path = Path('../configs/validation_rag.yaml')
with open(rag_path) as f:
    rag_config = yaml.safe_load(f)
pd_policy = rag_config['pd']

for metric_id in metric_ids:
    dev_value = result_dev[result_dev['metric_id'] == metric_id].iloc[0]['value']
    val_value = result_val[result_val['metric_id'] == metric_id].iloc[0]['value']
    rag_result = rag_for_metric(metric_id, pd_policy, dev_value, val_value)
    print(f"{metric_id.upper()} RAG Status: {rag_result['status']}")

# Plotting for validation results
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

roc_path = plot_roc(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='ROC Curve - Validation Data',
    out_path=plot_dir / 'roc_validation.png'
)
gini_path = plot_gini(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Gini Curve - Validation Data',
    out_path=plot_dir / 'gini_validation.png'
)
ks_path, _ = plot_ks_cdf_with_maxgap(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='KS CDF - Validation Data',
    out_path=plot_dir / 'ks_validation.png'
)

display(Image(filename=str(roc_path)))
display(Image(filename=str(gini_path)))
display(Image(filename=str(ks_path)))